In [1]:
import os, subprocess

REPO_URL = "https://github.com/Josef-Zayan/Run-the-Starter-Notebooks"
REPO_DIR = "/content/Run-the-Starter-Notebooks"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

print("Working dir:", os.getcwd())

Working dir: /content/Run-the-Starter-Notebooks


# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Josef-Zayan/Run-the-Starter-Notebooks/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

This is a ranking task built on top of a classifier.

Under the hood, the model answers a yes/no question for every page: is this page declining? Instead of a hard yes/no, it outputs a probability (e.g. 0.95, 0.80, 0.10).

But the editor doesn't need 16,262 "yes" answers — they only have time for about 50 pages. So the probabilities are used to sort the pages, and the editor reviews the top of the list first. The action this supports: an editor opens the top-ranked pages and decides whether to refresh, expand, or leave each one.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

Target: is_declining_label — 1 if the page is declining, 0 if not.

It comes from trend_direction, which compares impressions in the last 30 days to the previous 30 days. A drop of more than 20% counts as "down", and "down" becomes label = 1. Example from the data: 987 → 578 impressions is a 41.4% drop, so label = 1; 4206 → 3626 is only a 13.8% drop, so it counts as "stable" and label = 0.

This is a proxy, not a perfect target. The 20% cutoff is a defined rule, not a law of nature — a different cutoff would give different labels. It also describes pages that are already declining in the same window, not a prediction of future decline. Because the label is built from trend_direction and trend_pct, those two columns can never be used as model features — that would be leakage (the answer hidden inside the input).

In [2]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

print(df["trend_direction"].value_counts())
print()
print(df["is_declining_label"].value_counts())
print(f"Declining rate: {df['is_declining_label'].mean():.1%}")
print()
print(df[["impressions_prev_30d", "impressions_last_30d", "trend_pct",
          "trend_direction", "is_declining_label"]].head(5).to_string())

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

is_declining_label
1    16262
0    13738
Name: count, dtype: int64
Declining rate: 54.2%

   impressions_prev_30d  impressions_last_30d  trend_pct trend_direction  is_declining_label
0                   987                   578      -41.4            down                   1
1                  5915                  2501      -57.7            down                   1
2                  6089                  2382      -60.9            down                   1
3                  4206                  3626      -13.8          stable                   0
4                  6452                  4211      -34.7            down                   1


## 3. Success metric

Metric: Precision@50 — of the top 50 pages the model ranks first, what fraction are actually declining.

Why 50: an editor can only review a limited number of pages, so what matters is the quality of the top of the list, not the whole ranking.

What "good" means: the first bar is random ordering. On held-out clients (pages from clients the model never saw in training), 39.1% of pages are declining, so a random ordering scores about 0.39. Any useful model must clearly beat that. For reference, in notebook 01 the hand-written rule scored 0.24 — actually worse than random — while the random forest scored 0.74, roughly 1.9x random. So "good" for this lane means a Precision@50 well above 0.39 on held-out clients, and ideally matching or beating the random forest's 0.74 with a model I can still explain.

In [3]:
import numpy as np

clients = df["client_id"].astype(str).drop_duplicates().to_numpy()
shuffled = np.random.default_rng(42).permutation(clients)
test_clients = set(shuffled[:round(len(shuffled) * 0.2)])
test = df[df["client_id"].astype(str).isin(test_clients)]

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

rng = np.random.default_rng(0)
random_p50 = np.mean([precision_at_k(rng.random(len(test)), test["is_declining_label"]) for _ in range(1000)])

print(f"Held-out clients: {test['client_id'].nunique()} clients, {len(test)} pages")
print(f"Declining rate in held-out pages (base rate): {test['is_declining_label'].mean():.3f}")
print(f"Precision@50 of a RANDOM ordering (avg of 1,000 shuffles): {random_p50:.3f}")

Held-out clients: 6 clients, 2325 pages
Declining rate in held-out pages (base rate): 0.391
Precision@50 of a RANDOM ordering (avg of 1,000 shuffles): 0.390


## 4. The unit of analysis, as a real dataframe

Unit of analysis: one row = one page (one content item).

The code below confirms it: 30,000 rows and 30,000 unique content_id values, so no page appears twice. This matches the decision — the model scores pages, and the editor reviews pages.

Each page belongs to one of 32 pseudonymized clients, but clients are very unbalanced (from 3 pages to 7,008 pages per client). That's why evaluation holds out whole clients instead of random rows: otherwise pages from a big client would appear in both training and testing, and the score would look better than it really is on a new site.

In [4]:
print(f"Rows: {len(df)}  |  Columns: {df.shape[1]}")
print(f"Unique content_id: {df['content_id'].nunique()}  ->  one row = one page: {df['content_id'].is_unique}")

pages_per_client = df.groupby("client_id").size()
print(f"Clients: {df['client_id'].nunique()}")
print(f"Pages per client: min {pages_per_client.min()}, median {int(pages_per_client.median())}, max {pages_per_client.max()}")
print()

cols = ["content_id", "client_id", "content_type", "days_since_last_update",
        "impressions_90d", "avg_position", "ctr", "is_declining_label"]
df[cols].head(5)

Rows: 30000  |  Columns: 45
Unique content_id: 30000  ->  one row = one page: True
Clients: 32
Pages per client: min 3, median 567, max 7008



,content_id,client_id,content_type,days_since_last_update,impressions_90d,avg_position,ctr,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,20,3803,10.6,0.76,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,25,15320,20.3,0.05,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,20,12581,36.5,0.09,1
3,content_331d6c4de07b,client_19581e27de,keyword article,22,11751,6.2,0.49,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,14,19140,44.0,0.13,1


## 5. Why ML beats a fixed rule here

A fixed rule isn't enough here, and the code below shows why. I ranked held-out clients' pages by one signal at a time and measured Precision@50 (random ordering = 0.39, random forest from notebook 01 = 0.74):

- Intuition can point the wrong way. "Most stale pages first" sounds sensible, but it scored 0.22 — worse than random.
- The best single signal (worst avg_position first) scored 0.60. Useful, but still clearly below the random forest's 0.74. The model gains by combining several signals that no single if-statement captures.
- A rule tuned by hand doesn't transfer. The "stale x visible" rule from notebook 02 flags zero pages among the held-out clients — it simply doesn't fire on new sites.

So the pattern is real but spread across several interacting signals, and it changes from client to client. That's the case where a learned model earns its place over a hand-written rule.

One caveat: avg_position is measured over the same 90-day window as the decline label, so part of its strength may come from the decline itself rather than predicting it. I'll check this in the Week 3 leakage audit before relying on it.

In [5]:
def rule_precision_at_50(scores, labels):
    order = np.argsort(-np.asarray(scores), kind="stable")
    return np.asarray(labels)[order[:50]].mean()

y_test = test["is_declining_label"]
stale_visible = (test["days_since_last_update"] >= 180) & (test["impressions_90d"] >= 500)

single_signal_rules = {
    "Most stale first (days_since_last_update)": test["days_since_last_update"],
    "Oldest first (content_age_days)":           test["content_age_days"],
    "Most visible first (impressions_90d)":      test["impressions_90d"],
    "Worst position first (avg_position)":       test["avg_position"],
    "Lowest CTR first (ctr)":                    -test["ctr"],
}

print("Precision@50 on held-out clients (random ordering = 0.39, random forest = 0.74)\n")
for name, scores in single_signal_rules.items():
    print(f"  {rule_precision_at_50(scores, y_test):.2f}   {name}")

print(f"\nPages the notebook-02 'stale x visible' rule flags among held-out clients: {stale_visible.sum()}")

Precision@50 on held-out clients (random ordering = 0.39, random forest = 0.74)

  0.22   Most stale first (days_since_last_update)
  0.48   Oldest first (content_age_days)
  0.28   Most visible first (impressions_90d)
  0.60   Worst position first (avg_position)
  0.32   Lowest CTR first (ctr)

Pages the notebook-02 'stale x visible' rule flags among held-out clients: 0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.